# Section 4. Agents

"An AI agent is a system that uses an LLM to decide the control flow of an application."
— Harrison Chase, co-founder of LangChain

In other words, an agent is a system that uses a language model to determine the control flow of an application.

In the context of LangChain, agents are components that use language models to make decisions about which actions to take and in what order. Unlike chains, where the sequence of actions is predefined, agents use the language model as a reasoning engine to dynamically decide the actions to be taken.

LangChain agents are designed to interact with various tools, process inputs, and generate meaningful outputs. Instead of providing static responses, agents follow a looped structure to perform reasoning and iterative steps until the desired output is reached.

This dynamic approach allows agents to handle complex queries and adapt to different scenarios, making them essential components for building more intelligent and adaptable AI applications.

In **current LangChain versions**, the main entry point for this is `create_agent`, which **builds on top of LangGraph internally**. Here, we will stay focused on the LangChain layer and use that higher-level API.


In [6]:
!pip install -U python-dotenv
!pip install -U langchain
!pip install -U langchain-openai
!pip install -U langchain-google-genai
!pip install -U langchain-community
!pip install -U langchain-experimental
!pip install -U wikipedia
!pip install -U numexpr
!pip install -U pandas
!pip install -U openrouter
!pip install -U langchain-openrouter
!pip install -U langchain-classic

In [2]:
import getpass
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter

def get_model_name(model_name, temperature=0):
    if model_name == "gemini": # https://ai.google.dev/gemini-api/docs/rate-limits?hl=pt-br
        if "GOOGLE_API_KEY" not in os.environ: # https://ai.google.dev/gemini-api/docs/api-key
            os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")
        llm = ChatGoogleGenerativeAI(
            # model="gemini-1.5-pro", # max 50 / dia
            model="gemini-1.5-flash", # max 1500 / dia
            temperature=temperature,
        )
    elif model_name == "openai":
        if "OPENAI_API_KEY" not in os.environ: # https://platform.openai.com
            os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature,
        )
    elif model_name == "openrouter": # https://openrouter.ai/workspaces/default/
        if not os.getenv("OPENROUTER_API_KEY"):
            os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

        llm = ChatOpenRouter(
            model="openrouter/free",
            api_key=os.environ["OPENROUTER_API_KEY"],
            temperature=temperature,
        )

    return llm

llm = get_model_name('openrouter')

## 4.1 Tools

Tools are external functions that agents can use to perform specific tasks. They work like “skills” that expand what the agent is capable of doing. A well-defined agent is aware of which tools are available and understands how to use them—that is, what inputs and outputs are expected for the function defining the tool.

LangChain offers many built-in tools that can simplify the implementation of agent pipelines. In this lesson, we'll look at some of them and then learn how to create a new tool.

Built-in LangChain tools: https://python.langchain.com/docs/integrations/tools/

Naturally, a Large Language Model (LLM) does not, by itself, have the knowledge needed to call functions or interact directly with the external environment. For this, it's essential that the LLM's output is well structured and follows a defined format that can be interpreted by the system.

To enable an LLM to act as an agent, we use a prompting technique known as ReAct (Reasoning + Acting), which we studied earlier. With this method, the LLM is able to reason through the task and, based on that, decide what action to take—such as calling a specific tool.



Now, in the cell below, we'll begin exploring how to define tools and create an agent capable of intelligently selecting and using tools before returning a response to us.

First, we import two important utilities:
* `load_tools`, which loads built-in LangChain tools—in this case, a calculator and a tool that accesses Wikipedia.

* `create_agent`, which is the **current** high-level way to build agents in LangChain.

Internally, `create_agent` uses LangGraph as its runtime, but for now we do not need to worry about those lower-level details. In this class, we will stay focused on the LangChain interface itself.


In [7]:
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools

tools = load_tools(["llm-math", "wikipedia"], llm=llm)

agent_executor = create_agent(
    model=llm,
    tools=tools,
)

Now, let's test our agent with a math question.

In the last cell, all the actions taken by the LLM to answer the prompt are available in the `"messages"` key of the return.

By analyzing these messages, we see that the first is our original prompt sent to the LLM. Then, it makes a `function_call` with the name `Calculator`. Right after that, the `Calculator` tool responds with the result of the calculation. Finally, the LLM uses this information to formulate the final answer.

We were able to see the sequence of actions taken by the LLM. However, the format isn't very clear, and the visualization can be a bit confusing, which makes debugging or understanding what happened more difficult.

Wouldn't it be great if there was some kind of IDE to track all of this in a more visual way?

In [8]:
query = "What is the 25 percent of 300?"
response = agent_executor.invoke({
    "messages": [{"role": "user", "content": query}]
})

# Show response
for msg in response["messages"]:
    print(msg)

content='What is the 25 percent of 300?' additional_kwargs={} response_metadata={} id='42985669-fcb2-47d0-a4f6-e4be0b96ac12'
content='' additional_kwargs={'reasoning_content': 'We need to compute 25% of 300. 25% = 0.25, 0.25*300 = 75. Use Calculator tool.\n', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to compute 25% of 300. 25% = 0.25, 0.25*300 = 75. Use Calculator tool.\n'}]} response_metadata={'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'id': 'gen-1776286601-vRXsfvVaNEnYShlqLkDO', 'created': 1776286601, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'} id='lc_run--019d92ee-8cf5-7121-843c-604e0153e088-0' tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '0.25*300'}, 'id': 'chatcmpl-tool-b80c8e5e4eb5802c', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 400, 'output_tokens': 77, 'total_tokens': 477, 'input_token_details

## 4.2 LangSmith

LangSmith is a platform created by the LangChain team with the goal of helping developers debug, monitor, evaluate, and improve applications based on language models (LLMs).

When working with complex agents and chains, it can be difficult to understand exactly what’s happening behind the model’s decisions. That’s where LangSmith comes in—as a sort of “lab” for experimentation and observability for LLMs.

Among its features, the most commonly used is tracing, which allows you to view in detail how a chain or agent executed a task—from the user input to tool calls and the final generated responses.

To use LangSmith, first create an account on the official site: [LangSmith](https://smith.langchain.com/). Then, generate a new API key from your account.

Activating LangSmith is very simple: just set the environment variable `LANGSMITH_TRACING` to `true`. With that, executions from LangChain and LangGraph will automatically be synced to LangSmith, in the project specified by the `LANGSMITH_PROJECT` variable.

In [ ]:
if "LANGSMITH_API_KEY" not in os.environ:
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "insper_llm"

In [9]:
query = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU)\
what book did he write?"
response = agent_executor.invoke({
    "messages": [{"role": "user", "content": query}]
})

# Show response
for msg in response["messages"]:
    print(msg)

/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


content='Tom M. Mitchell is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU)what book did he write?' additional_kwargs={} response_metadata={} id='e3b2ca95-76cb-4515-8a3c-fd3686b44d5e'
content='' additional_kwargs={'reasoning_content': 'We need to answer: "Tom M. Mitchell is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU) what book did he write?" So the user asks: what book did he write? We need to find the book authored by Tom M. Mitchell. Likely "Machine Learning" (the classic textbook). Tom Mitchell is known for "Machine Learning" (1997). He also wrote "The Machine Learning Book"? Actually his well-known book is "Machine Learning" (McGraw-Hill, 1997). Also "Data Mining: Concepts and Techniques"? That\'s by Han, etc. But Tom Mitchell wrote "Machine Learning" (1997) and also "The Machine Learning Book"? Let\'s verify via Wikipedia. Use the Wikipedia tool to search "Tom M. M

## 4.3 Python REPL Agent

The Python REPL Agent is a type of agent that has the ability to execute Python code in real time. It’s extremely useful when you want the LLM to handle tasks involving more complex logic, precise mathematical calculations, list manipulation, dates, or any other type of detailed programming.

REPL stands for:

Read–Eval–Print Loop — in other words, an environment where the code is read, executed, and the result is returned immediately.

The execution environment is controlled, but it's still important to remember: running arbitrary code generated by an LLM requires caution, especially in production environments, so always be careful when using REPL agents.

In this section, we will first use a **convenient helper from `langchain_experimental`**, and then repeat the same idea using the **main `create_agent` interface**.


In [13]:
from langchain_experimental.agents.agent_toolkits import create_python_agent
from langchain_experimental.tools.python.tool import PythonREPLTool

agent = create_python_agent(
    llm,
    tool=PythonREPLTool(),
    verbose=True,
    handle_parsing_errors=True,
)

customer_list = [["Harrison", "Chase"],
                 ["Lang", "Chain"],
                 ["Dolly", "Too"],
                 ["Elle", "Elem"],
                 ["Geoff","Fusion"],
                 ["Trance","Former"],
                 ["Jen","Ayai"]
                ]

agent.invoke(f"""Sort these customers by \
last name and then first name \
and print the output: {customer_list}""")



> Entering new AgentExecutor chain...


Action: Python_REPL
Action Input: |
    data = [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]
    sorted_data = sorted(data, key=lambda x: (x[1], x[0]))
    print(sorted_data)
Observation: SyntaxError('invalid syntax', ('<string>', 1, 1, '|\n', 1, 2))
Thought:The task requires sorting customers by last name and then first name. The input list is `[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]`. Sorting by last name (second element) and first name (first element) yields:  
- Last name 'Chase' → 'Harrison'  
- Last name 'Chain' → 'Lang'  
- Last name 'Too' → 'Dolly'  
- Last name 'Elem' → 'Elle'  
- Last name 'Fusion' → 'Geoff'  
- Last name 'Former' → 'Trance'  
- Last name 'Ayai' → 'Jen'  

Thus, the sorted output is `[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff',

{'input': "Sort these customers by last name and then first name and print the output: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]",
 'output': "\\boxed{[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']}"}

From the execution, we can see that the agent first generated Python code to solve the problem. Then, that code was executed in the REPL environment, and the resulting output was returned to us, the user.

### 4.3.1 PythonREPL + create_agent

Now, let’s repeat the same process using `create_agent`.

This time, we start by defining the `PythonREPL` object and then associating it with a `Tool`:

```
# Tool
python_repl = PythonREPL()
print(python_repl.run("print(1+1)"))
tools = [Tool(
    name="python_repl",
    description="A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you should print it out with `print(...)`.",
    func=python_repl.run,
)]
```

After that, we define the system prompt and create the agent.

It’s worth noting that tool-calling quality can still vary by provider and model, so some combinations may be more reliable than others for code-execution demos.


In [19]:
from langchain.agents import create_agent
from langchain_experimental.utilities import PythonREPL
from langchain_core.tools import Tool
from langchain_core.globals import set_debug

# Tool
python_repl = PythonREPL()
tools = [
    Tool(
        name="python_repl",
        description=(
            "A Python shell. Use this to execute python commands. "
            "Input should be a valid python command. "
            "If you want to see the output of a value, print it with print(...)."
        ),
        func=python_repl.run,
    )
]

# System message
system_message = "You are a helpful assistant. Use the tools to answer user questions."

# Modern LangChain v1 agent
python_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_message,
)

Besides LangSmith, another way to view an execution in detail is to temporarily enable LangChain's global debug mode, as shown in the cell below:

In [20]:
customer_list = [
    ["Harrison", "Chase"],
    ["Lang", "Chain"],
    ["Dolly", "Too"],
    ["Elle", "Elem"],
    ["Geoff", "Fusion"],
    ["Trance", "Former"],
    ["Jen", "Ayai"],
]

message = f"""
Use the Python tool to sort this list of customers by last name, then first name.
Assume each item is [first_name, last_name].
Return only the final sorted list.

{customer_list}
"""

set_debug(True)
response = python_agent.invoke(
    {"messages": [{"role": "user", "content": message}]}
)
set_debug(False)

print(response["messages"][-1].content)

[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "\nUse the Python tool to sort this list of customers by last name, then first name.\nAssume each item is [first_name, last_name].\nReturn only the final sorted list.\n\n[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]\n"
    }
  ]
}
[chain/start] [chain:LangGraph > chain:model] Entering Chain run with input:
[inputs]
[llm/start] [chain:LangGraph > chain:model > llm:ChatOpenRouter] Entering LLM run with input:
{
  "prompts": [
    "System: You are a helpful assistant. Use the tools to answer user questions.\nHuman: \nUse the Python tool to sort this list of customers by last name, then first name.\nAssume each item is [first_name, last_name].\nReturn only the final sorted list.\n\n[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusi

## 4.5 How To Create Tools

LangChain already provides a wide range of ready-to-use tools—external APIs, databases, browsers, etc. That’s why it’s highly recommended to explore the full list of available tools, along with usage examples, directly in the official LangChain documentation: [Tools](https://docs.langchain.com/oss/python/langchain/tools).

In many situations, we need to create custom tools to meet specific needs in our agent applications. The simplest way to do this is by using the `@tool` decorator.

By default, the decorator uses the function name as the tool name, but this can be overridden by passing a string as the first argument. In addition, it uses the function’s docstring as the tool’s description — so it’s mandatory to provide an appropriate docstring.

In the cell below, we define a simple function called `time`, which returns the current time. As mentioned, when using the `@tool` decorator, it’s important to define both the docstring and input/output type annotations, as this information is automatically passed to the agent so it knows what the tool does and how to use it.

Finally, we print the tool’s name, parameters, and description, showing exactly how the information is exposed to the model.


In [23]:
from langchain.tools import tool
from datetime import date

@tool
def time(text: str) -> str:
    """Returns todays date, use this for any     questions related to knowing todays date.     The input should always be an empty string,     and this function will always return todays     date - any date mathmatics should occur     outside this function."""
    return str(date.today())


print(time.name)
print(time.description)
print(time.args)

time
Returns todays date, use this for any     questions related to knowing todays date.     The input should always be an empty string,     and this function will always return todays     date - any date mathmatics should occur     outside this function.
{'text': {'title': 'Text', 'type': 'string'}}


In [25]:
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools

system_prompt = "You are a helpful assistant, always start an answer with 'Bro...' and end with 'got it?'"

tools = load_tools(["llm-math", "wikipedia"], llm=llm)
tools = tools + [time]

agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

query = "whats the date today?"
response = agent_executor.invoke({
    "messages": [{"role": "user", "content": query}]
})

# Show response
for msg in response["messages"]:
    print(msg)

content='whats the date today?' additional_kwargs={} response_metadata={} id='3ed07a45-7b31-41d0-9652-435ae207c7ef'
content='' additional_kwargs={} response_metadata={'model_name': 'arcee-ai/trinity-large-preview:free', 'id': 'gen-1776287659-6VEzRps4jtQX9q8wD1XY', 'created': 1776287659, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'} id='lc_run--019d92fe-b3b3-71d0-ba67-6220bc1540b7-0' tool_calls=[{'name': 'time', 'args': {'text': ''}, 'id': 'call-c08d2b7e-5010-4071-b6eb-d930a592a3ad', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 328, 'output_tokens': 18, 'total_tokens': 346, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}
content='2026-04-15' name='time' id='74724584-ebb7-475d-a4aa-524fbb342fc6' tool_call_id='call-c08d2b7e-5010-4071-b6eb-d930a592a3ad'
content="Bro... today's date is2026-04-15. got it?" additional_kwargs={'reasoning_conten

## 4.4 Pandas Dataframe Agent

Now lets create a Pandas Pandas Dataframe Agent, using Python REPL and a custom tool.


In [34]:
import pandas as pd

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_experimental.utilities import PythonREPL
from langchain_core.globals import set_debug

# Load the CSV once
CSV_URL = "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv"
df = pd.read_csv(CSV_URL)

# Create a REPL instance
python_repl = PythonREPL()

@tool
def csv_python_repl(code: str) -> str:
    """Execute Python code over a preloaded pandas DataFrame named df.

    The DataFrame is already available as `df`.
    Use print(...) if you want to see values in the output.
    """
    preamble = """
import pandas as pd
"""
    # Recreate df inside the REPL session for consistency
    # This avoids depending on PythonREPL constructor-specific globals support
    load_df = f"df = pd.read_csv({CSV_URL!r})\n"
    full_code = preamble + load_df + code
    return python_repl.run(full_code)

system_message = """
You are a helpful data analysis assistant.

You have access to one Python tool called `csv_python_repl`.
Use it whenever the user asks questions about the CSV data.

Important:
- The pandas DataFrame is already available as `df`.
- Use Python to inspect the data.
- If you want to show a value, use print(...).
- Prefer short Python snippets.
- Return a concise final answer after using the tool.
"""

csv_agent = create_agent(
    model=llm,
    tools=[csv_python_repl],
    system_prompt=system_message,
)

set_debug(True)

response = csv_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the male / female rate?"
        }
    ]
})

set_debug(False)

print(response["messages"][-1].content)

[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "What is the male / female rate?"
    }
  ]
}
[chain/start] [chain:LangGraph > chain:model] Entering Chain run with input:
[inputs]
[llm/start] [chain:LangGraph > chain:model > llm:ChatOpenRouter] Entering LLM run with input:
{
  "prompts": [
    "System: \nYou are a helpful data analysis assistant.\n\nYou have access to one Python tool called `csv_python_repl`.\nUse it whenever the user asks questions about the CSV data.\n\nImportant:\n- The pandas DataFrame is already available as `df`.\n- Use Python to inspect the data.\n- If you want to show a value, use print(...).\n- Prefer short Python snippets.\n- Return a concise final answer after using the tool.\n\nHuman: What is the male / female rate?"
  ]
}
[llm/end] [chain:LangGraph > chain:model > llm:ChatOpenRouter] [12.97s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "",
    

## 4.5.1 More On Tool Creation

There are other ways to define a custom tool—check out the LangChain documentation to learn more:

[How to create tools | LangChain](https://docs.langchain.com/oss/python/langchain/tools)
